# V57 - Regularization sweep and need-gap features

Pre-registered plan: alpha was fixed at 3.0 without evidence, so sweep it first on the word-only baseline. Then test need-gap features from the earlier finding that skill correlates negatively with relevance: readiness minus prerequisite per module, plus chat volume statistics. Each candidate faces the word-only baseline under a fresh seed with the usual floor and worst-fold locks. At most one adoption, the best passer.

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 707
N_SPLITS = 5
GATE_FLOOR = 0.002
rng = np.random.RandomState(SEED)
WORDS = {"nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5, "tidak": 0, "ya": 5}

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v57")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

## Readiness, prerequisites, and chat stats

Prerequisite text maps to an ordinal. Need-gap is readiness minus prerequisite per module: positive means overqualified, negative means need. Chat stats are volume only, no text.

In [ ]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv", parse_dates=["timestamp"])
assess = pd.read_csv(DATA / "user_assessments.csv")
catalog = pd.read_csv(DATA / "modules_catalog.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
all_users = list(train["user_id"]) + test_ids

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float(WORDS.get(str(x).strip().lower(), np.nan))

blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})
Araw = np.array([[num(blobs[u].get(q)) for q in questions] for u in all_users])
Araw = np.where(np.isnan(Araw), np.nanmean(Araw, axis=0), Araw)
readiness = Araw.mean(1)

def prereq_ord(s):
    s = str(s).lower()
    if "menengah" in s:
        return 1.0
    if "lanjut" in s or "mahir" in s or "atas" in s:
        return 2.0
    return 0.0

pmap = {m: prereq_ord(catalog.set_index("module_id").loc[m, "prerequisite_level"]) if m in set(catalog["module_id"]) else 1.0 for m in MODS}
print("prereq map:", pmap)
P = np.array([pmap[m] for m in MODS])
GAP = readiness[:, None] - P[None, :] * 2.5
GAP = (GAP - GAP[:n_train].mean(0)) / (GAP[:n_train].std(0) + 1e-9)

g = chats.groupby("user_id")
STATS = np.array([[len(g.get_group(u)) if u in g.groups else 0,
                     (g.get_group(u)["timestamp"].max() - g.get_group(u)["timestamp"].min()).days if u in g.groups and len(g.get_group(u)) > 1 else 0,
                     np.mean([len(str(t)) for t in g.get_group(u)["user_chat_text"]]) if u in g.groups else 0] for u in all_users], float)
STATS = (STATS - STATS[:n_train].mean(0)) / (STATS[:n_train].std(0) + 1e-9)

A = (Araw - Araw[:n_train].mean(0)) / (Araw[:n_train].std(0) + 1e-9)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()
W_tr = W[:n_train]

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

## Sweep alpha, then gate feature additions

Alpha sweep runs on the baseline alone. Feature candidates stack onto the baseline with the swept alpha. One adoption at most.

In [ ]:
def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

def ridge_oof(Xtr, alpha):
    oof = np.zeros_like(Y)
    fids = np.zeros(len(Xtr), int)
    skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    for fi, (tr, va) in enumerate(skf.split(Xtr, strata, fold_groups)):
        for j in range(len(MODS)):
            m = Ridge(alpha=alpha, random_state=SEED)
            m.fit(Xtr[tr], Y[tr, j])
            oof[va, j] = m.predict(Xtr[va])
        fids[va] = fi + 1
    return oof, fids

oof3, fid = ridge_oof(W_tr, 3.0)
base = ndcg5(Y, oof3)
print(f"alpha=3.0: {base.mean():.5f}")
best_alpha, best_oof, best_mean = 3.0, oof3, base.mean()
for a in [0.5, 1.0, 10.0, 30.0]:
    oo, _ = ridge_oof(W_tr, a)
    mu = ndcg5(Y, oo).mean()
    print(f"alpha={a}: {mu:.5f}")
    if mu > best_mean:
        best_alpha, best_oof, best_mean = a, oo, mu
print("swept alpha:", best_alpha, f"{best_mean:.5f}")

cands = {"+ need-gap": np.hstack([W_tr, GAP[:n_train]]),
         "+ chat stats": np.hstack([W_tr, np.nan_to_num(STATS[:n_train])]),
         "+ gap + stats": np.hstack([W_tr, GAP[:n_train], np.nan_to_num(STATS[:n_train])])}
best = ("baseline", 0.0)
for name, Xtr in cands.items():
    oo, _ = ridge_oof(Xtr, best_alpha)
    d = ndcg5(Y, oo) - ndcg5(Y, best_oof)
    lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
    wf = min(ndcg5(Y[fid == k], oo[fid == k]).mean() - ndcg5(Y, best_oof)[fid == k].mean() for k in range(1, N_SPLITS + 1))
    ok = (lo > GATE_FLOOR) and (wf >= 0)
    print(f"{name}: delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f} {'ADOPT' if ok else 'reject'}")
    if ok and d.mean() > best[1]:
        best = (name, d.mean())
print("adopted:", best[0])